# Commutator heading estimate (`pirouette_data.kinematics`)

This notebook builds the pose + Harp-time dataset (`pirouette_data.ingestion`) and then appends a
heading-angle estimate derived from the commutator `AccumulatedCommutatorTurns` stream
(`pirouette_data.kinematics`).

Heading pipeline:
1. Subtract the first accumulated-turns value within the pose time window (reference orientation).
2. Convert turns to degrees (`* 360`) and wrap to `[0, 360)`.
3. Apply an offset so **facing right = 0°** (calibrated from the ear keypoints at the first frame
   where both ears are tracked).
4. Interpolate (`interp1d`) from the ~10 Hz commutator timeline onto the ~60 Hz camera timestamps.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pirouette_data import ingestion, kinematics

pd.set_option("display.max_columns", 40)

In [ ]:
POSE_DIR = r"C:/Users/brandon.pratt/Desktop/data/body-kinematics/pose_data"
S3_VIDEO_URI = "s3://aind-open-data/854393_2026-06-09_19-34-26/behavior-videos"
S3_BEHAVIOR_URI = "s3://aind-open-data/854393_2026-06-09_19-34-26/behavior"

## Build the pose dataset, then append heading

In [ ]:
df = ingestion.build_dataset(POSE_DIR, S3_VIDEO_URI)
print("pose dataset:", df.shape)

In [ ]:
# Inspect the ear-based offset calibration on its own
offset = kinematics.heading_offset_from_ears(df)
cal_frame = kinematics.first_valid_ear_frame(df)
print(f"calibration frame (first with both ears): row {cal_frame}")
print(f"heading offset (facing-right = 0°): {offset:.3f}°")
print("\nIf the heading ends up 180° out, re-run with forward_sign=-1;")
print("if it runs the wrong way in time, pass direction=-1 to append_commutator_heading.")

In [ ]:
df = kinematics.append_commutator_heading(df, S3_BEHAVIOR_URI)
df[["source_file", "harp_time", "time_since_start", "datetime_pacific", "commutator_heading_deg"]].head()

## Sanity checks

In [ ]:
h = df["commutator_heading_deg"]
print(f"range: [{h.min():.2f}, {h.max():.2f}] deg   NaNs: {h.isna().sum()}")
print(f"heading at frame 0: {h.iloc[0]:.3f}°  (== offset, since net turns = 0 at the reference)")

In [ ]:
# Heading over the first ~2 minutes (native camera rate)
window = df[df["time_since_start"] < df["time_since_start"].iloc[0] + 120]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(window["time_since_start"], window["commutator_heading_deg"], lw=0.6)
ax.set(
    xlabel="time since experiment start (s)",
    ylabel="heading (deg)",
    title="Commutator heading estimate (first 2 min)",
    ylim=(0, 360),
    yticks=[0, 90, 180, 270, 360],
)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of heading over the whole session (polar histogram)
theta = np.deg2rad(df["commutator_heading_deg"].dropna().to_numpy()[::50])
counts, edges = np.histogram(theta, bins=72, range=(0, 2 * np.pi))
centers = (edges[:-1] + edges[1:]) / 2

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111, projection="polar")
ax.bar(centers, counts, width=np.diff(edges), align="center", alpha=0.8)
ax.set_theta_zero_location("E")  # 0 deg = right
ax.set_theta_direction(1)  # counter-clockwise positive
ax.set_title("Heading occupancy (0° = facing right)")
plt.tight_layout()
plt.show()